# 05 — From a complete PDF to an evidence-grounded answer

**Optional extension: implemented workflow, unmeasured integrated answer score.** The component-level portfolio release is complete. Its pending integrated GPU attempt was stopped at closeout; no additional run is required. By default it runs entirely from verified public aggregates: no login, private PDF, model download, or paid GPU is needed to inspect it. A missing end-to-end result is shown as **not yet measured**, never filled in with oracle scores or synthetic tests.

The engineering question is specific: can the selected 9B reader answer from five pages selected by multilingual BM25, instead of being handed the correct evidence? The reader and semantic judge remain pinned. This is a fixed inference experiment, **not training or fine-tuning**.

## 1. Establish the evidence boundary

In [1]:
import json
from pathlib import Path

from IPython.display import HTML, display

from lava.evaluation.reporting import load_report
from lava.evaluation.system import load_summary
from lava.evaluation.walkthrough import TABLE_STYLE, render_table
from lava.notebook_support import find_repo_root
from lava.readers.runtime_logging import RuntimeEventLogger
from lava.retrieval.pipeline import load_public_report
from lava.retrieval.reporting import recall_chart

ROOT = find_repo_root(Path.cwd())
logger = RuntimeEventLogger("notebook.system")
with logger.stage("01_verify_evidence", heartbeat_seconds=15):
    readers = load_report(ROOT)
    baseline = next(
        row for row in readers["current_models"] if row["model_key"] == "qwen35_9b_fused_direct"
    )
    assert baseline["complete"] and baseline["semantic_summary"]["contract_current"]
    oracle = baseline["semantic_summary"]["metrics"]
    retrieval = load_public_report(ROOT)
    result = load_summary(ROOT)
    status = result["status"] if result else "End-to-end inference not yet measured"
    display(
        HTML(
            TABLE_STYLE
            + render_table(
                [
                    {"Evidence": "Current end-to-end status", "Observation": status},
                    {
                        "Evidence": "Oracle baseline",
                        "Observation": f"{oracle['question_micro']['overall']:.2%} local LAVA; correct pages supplied",
                    },
                    {
                        "Evidence": "Labeled pilot",
                        "Observation": "16 previously examined questions / 5 PDFs / 15 Japanese + 1 Vietnamese",
                    },
                    {
                        "Evidence": "Claim boundary",
                        "Observation": "Training diagnostic; no hidden-test, organizer-server or state-of-the-art claim",
                    },
                ],
                caption="Measured evidence, not promised performance",
            )
        )
    )

{"component": "notebook.system", "elapsed_seconds": 0.0, "event": "01_verify_evidence.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-08T03:17:33.999+00:00"}


Evidence,Observation
Current end-to-end status,End-to-end inference not yet measured
Oracle baseline,87.02% local LAVA; correct pages supplied
Labeled pilot,16 previously examined questions / 5 PDFs / 15 Japanese + 1 Vietnamese
Claim boundary,"Training diagnostic; no hidden-test, organizer-server or state-of-the-art claim"


{"component": "notebook.system", "elapsed_seconds": 0.373, "event": "01_verify_evidence.completed", "level": "INFO", "stage_elapsed_seconds": 0.372, "timestamp_utc": "2026-09-08T03:17:34.371+00:00"}


## Run this workflow yourself

The controls below belong to this notebook. **Run All with the defaults only reads the published analysis.** Nothing is submitted to Kaggle and no new GPU is allocated. Reporting/CI kernels additionally enforce an offline mode even when an edited notebook was saved with run controls enabled.

To measure the 16-question integrated pilot, set `RUN_PILOT = True` and `ACKNOWLEDGE_AWS_CHARGES = "YES"`, then run the control cell and the remaining analysis. The original attempt 1 was stopped at closeout. A future deliberate retry uses `PILOT_ATTEMPT = 2` and `ALLOW_PAID_RETRY = True`. Keep the same attempt number when reconnecting to that new run; only a deliberately approved new attempt uses `ALLOW_PAID_RETRY = True`. The existing completed reader sweeps and retrieval checkpoints are reused. A notebook action never recursively republishes the notebook currently running.

The pilot uses one managed GPU and a 1,800-second runtime limit. Its $5 estimate guard is not an account-wide spending cap; Studio, storage and logs are separate. Real model failures are retained, not hidden to improve the score.

In [2]:
from lava.notebook_operations import download_link, run_operation

RUN_PILOT = False
RUN_TEST_INFERENCE = False
EXPORT_SAVED_TEST = False
ACKNOWLEDGE_AWS_CHARGES = "NO"
PILOT_ATTEMPT = 1
TEST_ATTEMPT = 1
ALLOW_PAID_RETRY = False
PILOT_TRAINING_ESTIMATE_LIMIT_USD = 5.0
TEST_TRAINING_ESTIMATE_LIMIT_USD = 15.0

pilot_action = run_operation(
    ROOT,
    "pilot",
    enabled=RUN_PILOT,
    acknowledge_charges=ACKNOWLEDGE_AWS_CHARGES,
    attempt=PILOT_ATTEMPT,
    retry=ALLOW_PAID_RETRY,
    maximum_training_usd=PILOT_TRAINING_ESTIMATE_LIMIT_USD,
)
result = load_summary(ROOT)
display(HTML(render_table([pilot_action], caption="Your pilot execution control")))

{"cloud_access": false, "component": "notebook.operation", "elapsed_seconds": 0.0, "event": "notebook.operation.disabled", "level": "INFO", "operation": "pilot", "timestamp_utc": "2026-09-08T03:17:34.379+00:00"}


operation,status,uploaded_to_kaggle
pilot,disabled,False


## 2. Design useful inputs, not more features indiscriminately

For this document-AI task, feature engineering means **preserving the information the answer depends on**. Native text captures searchable words and numbers; page images retain tables, charts, and spatial relationships; explicit physical page numbers support auditable citations. More context can also introduce distractors and consume memory.

The frozen retrieval representation uses Unicode NFKC normalization, words, and character bigrams/trigrams. This lets Japanese match without whitespace tokenization while retaining Vietnamese diacritics. Frequencies and page-length normalization are computed within the queried PDF. Gold answers and gold page labels never enter the reader request.

The five-page budget was fixed for this final pilot **after examining the earlier retrieval diagnostics**; it is a development choice, not a held-out selection. Ten pages retrieved all known evidence but have not been demonstrated to improve answer quality or latency. We do not equate retrieval recall with answer accuracy.

In [3]:
with logger.stage("02_inspect_input_tradeoff", heartbeat_seconds=15):
    display(HTML(recall_chart(retrieval, "all_evidence_at_k")))
    display(
        HTML(
            render_table(
                [
                    {
                        "Input decision": "PDF text + rendered image",
                        "Reason": "Lexical matching plus visual tables/layout; no destructive text-only assumption",
                    },
                    {
                        "Input decision": "Five ranked pages, shown in physical order",
                        "Reason": "Bounded context; stable page identities instead of renumbered citations",
                    },
                    {
                        "Input decision": "Pinned source and model hashes",
                        "Reason": "Detect changed inputs; reuse only compatible answers",
                    },
                    {
                        "Input decision": "Labels separated from ReaderInput",
                        "Reason": "Reference answers and pages become visible only in the evaluator",
                    },
                ],
                caption="Information-preserving input design",
            )
        )
    )

{"component": "notebook.system", "elapsed_seconds": 0.387, "event": "02_inspect_input_tradeoff.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-08T03:17:34.385+00:00"}


Input decision,Reason
PDF text + rendered image,Lexical matching plus visual tables/layout; no destructive text-only assumption
"Five ranked pages, shown in physical order",Bounded context; stable page identities instead of renumbered citations
Pinned source and model hashes,Detect changed inputs; reuse only compatible answers
Labels separated from ReaderInput,Reference answers and pages become visible only in the evaluator


{"component": "notebook.system", "elapsed_seconds": 0.389, "event": "02_inspect_input_tradeoff.completed", "level": "INFO", "stage_elapsed_seconds": 0.003, "timestamp_utc": "2026-09-08T03:17:34.388+00:00"}


## 3. Follow the actual execution path

Frozen source versions → reuse full-PDF text extraction → reuse label-blind BM25 rankings → render only selected unique pages → construct label-free `ReaderInput` → invoke the same pinned 9B reader → independently parse the exact generation → persist and read back each answer → evaluate with the unchanged semantic judge.

The original oracle experiment retains its strict gold-page-alignment contract. The new input type has no reference answer or gold evidence fields. Output citations must be members of the **supplied physical page set**, not the reference set. Invalid generations remain failures in the denominator.

A successful AWS job status is necessary but insufficient: complete question coverage, checksums, re-parsed outputs, judge controls, and public report integrity must also pass.

In [4]:
with logger.stage("03_inspect_frozen_execution", heartbeat_seconds=15):
    config = json.loads((ROOT / "configs/system_evaluation.json").read_text())
    display(
        HTML(
            render_table(
                [
                    {
                        "Contract": "Reader",
                        "Value": "Qwen3.5-9B / bfloat16 / direct deterministic decoding",
                    },
                    {"Contract": "Page budget", "Value": config["page_budget"]},
                    {
                        "Contract": "GPU attempt",
                        "Value": "One ml.g6e.2xlarge; 1,800-second server runtime cap; no endpoint",
                    },
                    {
                        "Contract": "Resume",
                        "Value": "Reattach to a deterministic job name; skip verified answer checkpoints",
                    },
                    {
                        "Contract": "New paid retry",
                        "Value": "Separate explicit approval; never automatic",
                    },
                ],
                caption="A bounded experiment, not an open-ended search",
            )
        )
    )

{"component": "notebook.system", "elapsed_seconds": 0.394, "event": "03_inspect_frozen_execution.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-08T03:17:34.393+00:00"}


Contract,Value
Reader,Qwen3.5-9B / bfloat16 / direct deterministic decoding
Page budget,5
GPU attempt,"One ml.g6e.2xlarge; 1,800-second server runtime cap; no endpoint"
Resume,Reattach to a deterministic job name; skip verified answer checkpoints
New paid retry,Separate explicit approval; never automatic


{"component": "notebook.system", "elapsed_seconds": 0.396, "event": "03_inspect_frozen_execution.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-08T03:17:34.395+00:00"}


## 4. Compare the official-formula components

The published LAVA formula averages semantic answer credit and evidence-page F1 per question. Unordered list answers use optimal semantic matching; ordered lists use semantic longest-common-subsequence credit. Our pinned local Gemma judge implements this evaluation structure, but the organizer's exact prompt/runtime is not published, so these are **local published-formula scores**, not official leaderboard scores.

The comparison below uses actual predicted evidence pages. Supplying the reference pages during scoring would incorrectly inflate the complete-system score. The oracle-to-retrieved difference includes missed evidence, distractor pages, and context-volume changes; it is not a pure causal estimate of retrieval alone.

In [5]:
def percent(value):
    return "Not yet measured" if value is None else f"{value:.2%}"


with logger.stage("04_compare_quality", heartbeat_seconds=15):
    current = result["metrics"]["question_micro"] if result else {}
    rows = [
        {
            "Metric": label,
            "Oracle pages": percent(oracle["question_micro"][key]),
            "Retrieved pages": percent(current.get(key)),
            "Change (points)": f"{100 * (current[key] - oracle['question_micro'][key]):+.2f}"
            if key in current
            else "Not yet measured",
        }
        for key, label in (
            ("answer", "Semantic answer credit"),
            ("grounding", "Evidence-page F1"),
            ("overall", "Local LAVA overall"),
        )
    ]
    display(HTML(render_table(rows, caption="Same 9B configuration and semantic judge")))
    if result:
        display(
            HTML(
                render_table(
                    [
                        {
                            "Weighting": "Question average",
                            "Oracle": percent(oracle["question_micro"]["overall"]),
                            "Retrieved": percent(current["overall"]),
                        },
                        {
                            "Weighting": "Equal-document average",
                            "Oracle": percent(oracle["document_macro"]["overall"]),
                            "Retrieved": percent(result["metrics"]["document_macro"]["overall"]),
                        },
                    ],
                    caption="Do not let PDFs with more questions hide inconsistent behavior",
                )
            )
        )

{"component": "notebook.system", "elapsed_seconds": 0.403, "event": "04_compare_quality.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-08T03:17:34.401+00:00"}


Metric,Oracle pages,Retrieved pages,Change (points)
Semantic answer credit,80.15%,Not yet measured,Not yet measured
Evidence-page F1,93.90%,Not yet measured,Not yet measured
Local LAVA overall,87.02%,Not yet measured,Not yet measured


{"component": "notebook.system", "elapsed_seconds": 0.404, "event": "04_compare_quality.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-08T03:17:34.403+00:00"}


## 5. Diagnose the errors before proposing another model

Each question is assigned one diagnostic category in this order: invalid response; missing required evidence; incomplete/incorrect answer despite complete evidence; incomplete citation; full credit. These categories are operational triage, not proof of causation. A missing page and an answer error can coexist.

Report every PDF and keep the single Vietnamese question visible as a sample-size limitation, not a language benchmark. The document-level paired interval is exploratory: only five PDFs were available, and all were previously examined during development.

In [6]:
with logger.stage("05_examine_failures", heartbeat_seconds=15):
    if result:
        display(
            HTML(
                render_table(
                    [
                        {"Failure category": category.replace("_", " "), "Questions": count}
                        for category, count in result["diagnostics"]["failure_counts"].items()
                    ],
                    caption="All 16 questions remain accounted for",
                )
            )
        )
        display(
            HTML(
                render_table(
                    [
                        {
                            "PDF": name,
                            "Questions": sum(
                                row["document"] == name for row in result["per_question"]
                            ),
                            "Oracle LAVA": percent(oracle["by_document"][name]["overall"]),
                            "Retrieved LAVA": percent(values["overall"]),
                            "Change (points)": f"{100 * (values['overall'] - oracle['by_document'][name]['overall']):+.2f}",
                        }
                        for name, values in result["metrics"]["by_document"].items()
                    ],
                    caption="Paired document consistency",
                )
            )
        )
        display(
            HTML(
                render_table(
                    [
                        {
                            "Question": row["question"],
                            "PDF": row["document"],
                            "Answer credit": percent(row["answer_score"]),
                            "Evidence F1": percent(row["evidence_f1"]),
                            "Diagnosis": row["failure_category"].replace("_", " "),
                        }
                        for row in result["per_question"]
                    ],
                    caption="Sanitized question diagnostics; no raw source data",
                )
            )
        )
    else:
        display(
            HTML(
                render_table(
                    [
                        {
                            "Analysis": "End-to-end error taxonomy",
                            "Result": "Awaiting actual retrieved-page generations",
                        },
                        {
                            "Analysis": "Known retrieval limitation",
                            "Result": "At k=5, complete evidence is missing for 2 of 16 questions",
                        },
                    ],
                    caption="What can and cannot be concluded now",
                )
            )
        )

{"component": "notebook.system", "elapsed_seconds": 0.412, "event": "05_examine_failures.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-08T03:17:34.410+00:00"}


Analysis,Result
End-to-end error taxonomy,Awaiting actual retrieved-page generations
Known retrieval limitation,"At k=5, complete evidence is missing for 2 of 16 questions"


{"component": "notebook.system", "elapsed_seconds": 0.413, "event": "05_examine_failures.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-08T03:17:34.412+00:00"}


## 6. Measure execution and preserve successful work

Generation latency, reader latency, and AWS billable time measure different things. Reader timing includes preprocessing and, for the first uncached question, model loading; it excludes retrieval, provisioning, and durable writes. Recovered answers can span multiple GPU attempts. Do not call these measurements full competition-runtime compliance.

Each operator invocation records UTC timestamps, stage/total elapsed time, and heartbeats. Exact generations, answer checkpoints, judge decisions, completed reports, and successful notebook publications are durable in private S3. Analysis reruns verify existing outputs instead of paying for new inference.

In [7]:
with logger.stage("06_inspect_runtime_and_recovery", heartbeat_seconds=15):
    if result:
        runtime = result["runtime"]
        display(
            HTML(
                render_table(
                    [
                        {"Measurement": name.replace("_", " "), "Value": value}
                        for name, value in runtime.items()
                    ],
                    caption="Measured reader runtime; scope is explicit",
                )
            )
        )
    else:
        display(
            HTML(
                render_table(
                    [
                        {
                            "Measurement": "Retrieved-page GPU runtime/memory",
                            "Value": "Not yet measured",
                        },
                        {
                            "Measurement": "Existing retrieval initial run",
                            "Value": f"{retrieval['first_run_seconds']:.3f} seconds",
                        },
                        {
                            "Measurement": "Resume guarantee",
                            "Value": "Verified completed checkpoints reused; an interrupted unsaved question may need repetition",
                        },
                    ],
                    caption="Measured timing versus unmeasured work",
                )
            )
        )
    prices = json.loads((ROOT / "reports/aws/training_prices.json").read_text())
    display(
        HTML(
            render_table(
                [
                    {
                        "Cost boundary": "Published price snapshot",
                        "Meaning": "See reports/aws/training_prices.json for dated Training rates",
                    },
                    {
                        "Cost boundary": "Explicit launch budget",
                        "Meaning": "Per-attempt compute estimate only; not a total AWS billing cap",
                    },
                    {
                        "Cost boundary": "Other charges",
                        "Meaning": "Existing Studio, storage, logs, data transfer and previous attempts are separate",
                    },
                ],
                caption="No misleading all-in cost claim",
            )
        )
    )

{"component": "notebook.system", "elapsed_seconds": 0.419, "event": "06_inspect_runtime_and_recovery.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-08T03:17:34.418+00:00"}


Measurement,Value
Retrieved-page GPU runtime/memory,Not yet measured
Existing retrieval initial run,8.375 seconds
Resume guarantee,Verified completed checkpoints reused; an interrupted unsaved question may need repetition


Cost boundary,Meaning
Published price snapshot,See reports/aws/training_prices.json for dated Training rates
Explicit launch budget,Per-attempt compute estimate only; not a total AWS billing cap
Other charges,"Existing Studio, storage, logs, data transfer and previous attempts are separate"


{"component": "notebook.system", "elapsed_seconds": 0.422, "event": "06_inspect_runtime_and_recovery.completed", "level": "INFO", "stage_elapsed_seconds": 0.003, "timestamp_utc": "2026-09-08T03:17:34.420+00:00"}


## Generate and download your own submission

**You generate the file here; you decide when and whether to upload it.** This is not the 16-row training diagnostic and it never uses the sample template's placeholder answers. The code reads the pinned test inputs, retrieves evidence for all **624 questions across 200 PDFs**, runs the same pinned 9B reader, validates every ID and physical page number, writes a UTF-8 CSV in template order, and displays a local download link. No code in this workflow invokes a Kaggle upload API.

Set `RUN_TEST_INFERENCE = True` and explicitly acknowledge charges in the control cell above, then execute this section. Test inference is a **separate paid operation**: one GPU, a 7,200-second runtime limit, a $15 per-attempt estimate guard (not an account-wide cap), and no automatic paid retry. Actual full-test throughput and competition eligibility are not yet verified. Do not start it while the pilot GPU job is running.

After completed inference, set `RUN_TEST_INFERENCE = False` and `EXPORT_SAVED_TEST = True` to reconstruct the file from durable predictions without a new GPU job. A disconnect reuses the same attempt and compatible checkpoints; an interrupted uncommitted question may repeat. Invalid or abstained answers are retained for private review and block a misleading 'complete' export rather than becoming invented answers. The existing `prepare_submission.py --mode build` interface also validates predictions you have reviewed independently.

The file appears at `artifacts/submission/submission.csv`; its checksum and provenance are in `manifest.json`, and the immutable bundle remains in private S3. The notebook output contains only a link—not embedded questions, answers, or CSV bytes. In Studio, you can also find that file in the file browser, right-click it, and select **Download**. Manual upload/competition acceptance is your separate action.

Before publishing the notebook on GitHub, restore run controls to `False` and charges to `"NO"`; `make notebooks` publishes the safe, analysis-only view. Do not commit private files from `artifacts/`.

In [8]:
test_action = run_operation(
    ROOT,
    "test",
    enabled=RUN_TEST_INFERENCE,
    acknowledge_charges=ACKNOWLEDGE_AWS_CHARGES,
    attempt=TEST_ATTEMPT,
    retry=ALLOW_PAID_RETRY,
    maximum_training_usd=TEST_TRAINING_ESTIMATE_LIMIT_USD,
)
export_action = run_operation(ROOT, "export", enabled=EXPORT_SAVED_TEST)
display(HTML(render_table([test_action, export_action], caption="Your test and export controls")))
display(HTML(download_link(ROOT)))

{"cloud_access": false, "component": "notebook.operation", "elapsed_seconds": 0.0, "event": "notebook.operation.disabled", "level": "INFO", "operation": "test", "timestamp_utc": "2026-09-08T03:17:34.425+00:00"}


{"cloud_access": false, "component": "notebook.operation", "elapsed_seconds": 0.0, "event": "notebook.operation.disabled", "level": "INFO", "operation": "export", "timestamp_utc": "2026-09-08T03:17:34.426+00:00"}


operation,status,uploaded_to_kaggle
test,disabled,False
export,disabled,False


## 7. Portfolio closeout and optional reproduction

**The completed release is the component-level research benchmark:** three measured
reader configurations, full-document retrieval, lexical feature research, the
visual challenger, validated reports, and six executed notebooks. The optional
integrated attempt was stopped during the September 8 closeout. No integrated
answer score, deployed application, full-test inference, or Kaggle result is claimed.

This notebook preserves the implemented workflow and its honest measurement status.
A future integrated benchmark would require 16 verified retrieved-page predictions,
the unchanged judge, complete metrics and failure analysis, and successful quality
checks. Those are acceptance criteria for a future extension, not unfinished work
in this release. The [operator guide](../docs/system_evaluation.md) retains explicit
cost and retry controls. No more experiments are required to present this portfolio.


In [9]:
with logger.stage("07_record_conclusion", heartbeat_seconds=15):
    if result:
        delta = result["question_mean_delta"]
        conclusion = (
            f"Retrieved-page local LAVA: {result['metrics']['question_micro']['overall']:.2%}; "
            f"change from oracle: {100 * delta:+.2f} percentage points. "
            "The integrated training pilot is measured. Retain its failures and limitations; publish this evidence rather than restarting model selection."
        )
    else:
        conclusion = (
            "Reader selection and retrieval diagnostics are measured; the integrated score is not. "
            "The component benchmark is complete. The optional attempt was stopped at closeout; no further run is required for this portfolio release."
        )
    display(HTML(render_table([{"Conclusion": conclusion}], caption="Final system evaluation")))
logger.emit("system.walkthrough.completed", measured=result is not None, question_count=16)

{"component": "notebook.system", "elapsed_seconds": 0.434, "event": "07_record_conclusion.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-08T03:17:34.433+00:00"}


Conclusion
Reader selection and retrieval diagnostics are measured; the integrated score is not. The component benchmark is complete. The optional attempt was stopped at closeout; no further run is required for this portfolio release.


{"component": "notebook.system", "elapsed_seconds": 0.436, "event": "07_record_conclusion.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-08T03:17:34.434+00:00"}


{"component": "notebook.system", "elapsed_seconds": 0.436, "event": "system.walkthrough.completed", "level": "INFO", "measured": false, "question_count": 16, "timestamp_utc": "2026-09-08T03:17:34.435+00:00"}
